# Лабораторная 1. Интерактивный анализ данных велопарковок SF Bay Area Bike Share в Apache Spark с использованием Spark SQL и DataFrame API

## Описание данных

https://www.kaggle.com/benhamner/sf-bay-area-bike-share

stations.csv схема:

```
id: station ID number
name: name of station
lat: latitude
long: longitude
dock_count: number of total docks at station
city: city (San Francisco, Redwood City, Palo Alto, Mountain View, San Jose)
installation_date: original date that station was installed. If station was moved, it is noted below.
```

trips.csv схема:

```
id: numeric ID of bike trip
duration: time of trip in seconds
start_date: start date of trip with date and time, in PST
start_station_name: station name of start station
start_station_id: numeric reference for start station
end_date: end date of trip with date and time, in PST
end_station_name: station name for end station
end_station_id: numeric reference for end station
bike_id: ID of bike used
subscription_type: Subscriber = annual or 30-day member; Customer = 24-hour or 3-day member
zip_code: Home zip code of subscriber (customers can choose to manually enter zip at kiosk however data is unreliable)
```

https://spark.apache.org/docs/latest/sql-programming-guide.html

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import pyspark.sql as sql

In [2]:
conf = SparkConf().setAppName("L1_interactive_bike_analysis").setMaster("yarn")

In [3]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('WARN')

# Пример чтения csv файлов и работы с дефектными данными

Список опций чтения и записи для CSV файлов https://spark.apache.org/docs/latest/sql-data-sources-csv.html#data-source-option

Формат паттерна временной метки Spark SQL отличается от python библиотеки datetime.
https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html

In [4]:
tripData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv("trips.csv")

tripData

DataFrame[id: int, duration: int, start_date: timestamp, start_station_name: string, start_station_id: int, end_date: timestamp, end_station_name: string, end_station_id: int, bike_id: int, subscription_type: string, zip_code: string]

In [5]:
tripData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)


In [6]:
tripData.show(n=5)

+------+----------+---------------------+-----------------------------------+------------------+---------------------+-----------------------------------+----------------+---------+-------------------+----------+
| id   | duration | start_date          | start_station_name                | start_station_id | end_date            | end_station_name                  | end_station_id | bike_id | subscription_type | zip_code |
+------+----------+---------------------+-----------------------------------+------------------+---------------------+-----------------------------------+----------------+---------+-------------------+----------+
| 4576 | 63       | 2013-08-29 14:13:00 | South Van Ness at Market          | 66               | 2013-08-29 14:14:00 | South Van Ness at Market          | 66             | 520     | Subscriber        | 94127    |
| 4607 | 70       | 2013-08-29 14:42:00 | San Jose City Hall                | 10               | 2013-08-29 14:43:00 | San Jose City Hall           

In [7]:
? tripData.dropna

In [8]:
tripData.dropna().show(n=5)

+------+----------+---------------------+-----------------------------------+------------------+---------------------+-----------------------------------+----------------+---------+-------------------+----------+
| id   | duration | start_date          | start_station_name                | start_station_id | end_date            | end_station_name                  | end_station_id | bike_id | subscription_type | zip_code |
+------+----------+---------------------+-----------------------------------+------------------+---------------------+-----------------------------------+----------------+---------+-------------------+----------+
| 4576 | 63       | 2013-08-29 14:13:00 | South Van Ness at Market          | 66               | 2013-08-29 14:14:00 | South Van Ness at Market          | 66             | 520     | Subscriber        | 94127    |
| 4607 | 70       | 2013-08-29 14:42:00 | San Jose City Hall                | 10               | 2013-08-29 14:43:00 | San Jose City Hall           

In [9]:
tripData.describe().show()

In [10]:
stationData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y')\
.csv("stations.csv")

stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: date (nullable = true)


In [11]:
stationData.show(n=5)

+----+-----------------------------------+-----------+-------------+------------+----------+-------------------+
| id | name                              | lat       | long        | dock_count | city     | installation_date |
+----+-----------------------------------+-----------+-------------+------------+----------+-------------------+
| 2  | San Jose Diridon Caltrain Station | 37.329732 | -121.901782 | 27         | San Jose | 2013-08-06        |
| 3  | San Jose Civic Center             | 37.330698 | -121.888979 | 15         | San Jose | 2013-08-05        |
| 4  | Santa Clara at Almaden            | 37.333988 | -121.894902 | 11         | San Jose | 2013-08-06        |
| 5  | Adobe on Almaden                  | 37.331415 | -121.893200 | 19         | San Jose | 2013-08-05        |
| 6  | San Pedro Square                  | 37.336721 | -121.894074 | 15         | San Jose | 2013-08-07        |
+----+-----------------------------------+-----------+-------------+------------+----------+----

In [12]:
stationData.describe().show()

# Пример использования DataFrame API

Выполните операцию объединения коллекций по ключу с помощью функции join. Объедините stationsIndexed и tripsByStartTerminals, stationsIndexed и tripsByEndTerminals.

https://spark.apache.org/docs/latest/sql-getting-started.html#untyped-dataset-operations-aka-dataframe-operations

In [13]:
tripData.printSchema()
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: date (nullable = true)


In [14]:
stationsView = stationData.select(stationData['id'], stationData['name'], stationData['lat'], stationData['long'])
stationsView.show()

+----+-----------------------------------+-----------+-------------+
| id | name                              | lat       | long        |
+----+-----------------------------------+-----------+-------------+
| 2  | San Jose Diridon Caltrain Station | 37.329732 | -121.901782 |
| 3  | San Jose Civic Center             | 37.330698 | -121.888979 |
| 4  | Santa Clara at Almaden            | 37.333988 | -121.894902 |
| 5  | Adobe on Almaden                  | 37.331415 | -121.893200 |
| 6  | San Pedro Square                  | 37.336721 | -121.894074 |
+----+-----------------------------------+-----------+-------------+


In [15]:
from pyspark.sql import functions as F

startTrips = tripData.select(
    tripData.id.alias("trip_id"),
    tripData.duration,
    tripData.start_station_id
).join(
    stationsView,
    tripData.start_station_id == stationsView.id,
    "inner"
).drop(stationsView.id)

startTrips.show()


+---------+----------+------------------+-----------------------------------+-----------+-------------+
| trip_id | duration | start_station_id | name                              | lat       | long        |
+---------+----------+------------------+-----------------------------------+-----------+-------------+
| 4576    | 63       | 66               | South Van Ness at Market          | 37.774814 | -122.418954 |
| 4607    | 70       | 10               | San Jose City Hall                | 37.337391 | -121.886995 |
| 4130    | 71       | 27               | Mountain View City Hall           | 37.389218 | -122.081896 |
| 4251    | 77       | 2                | San Jose Diridon Caltrain Station | 37.329732 | -121.901782 |
| 4299    | 83       | 66               | South Van Ness at Market          | 37.774814 | -122.418954 |
+---------+----------+------------------+-----------------------------------+-----------+-------------+


In [16]:
startTrips.show()

+---------+----------+------------------+-----------------------------------+-----------+-------------+
| trip_id | duration | start_station_id | name                              | lat       | long        |
+---------+----------+------------------+-----------------------------------+-----------+-------------+
| 4576    | 63       | 66               | South Van Ness at Market          | 37.774814 | -122.418954 |
| 4607    | 70       | 10               | San Jose City Hall                | 37.337391 | -121.886995 |
| 4130    | 71       | 27               | Mountain View City Hall           | 37.389218 | -122.081896 |
| 4251    | 77       | 2                | San Jose Diridon Caltrain Station | 37.329732 | -121.901782 |
| 4299    | 83       | 66               | South Van Ness at Market          | 37.774814 | -122.418954 |
+---------+----------+------------------+-----------------------------------+-----------+-------------+


In [17]:
tripData.printSchema()
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: date (nullable = true)


# Пример использования Spark SQL API

https://spark.apache.org/docs/latest/sql-getting-started.html#running-sql-queries-programmatically

In [18]:
stationData.createOrReplaceTempView("stations")
tripData.createOrReplaceTempView("trips")

https://spark.apache.org/docs/latest/sql-ref.html

In [19]:
endTrips = spark.sql("""
SELECT
    trips.id AS trip_id,
    trips.end_station_id,
    trips.duration,
    stations.name AS station_name,
    stations.lat,
    stations.long
FROM trips
INNER JOIN stations
    ON trips.end_station_id = stations.id
""")


In [20]:
endTrips.show()

+---------+----------------+----------+-----------------------------------+-----------+-------------+
| trip_id | end_station_id | duration | station_name                      | lat       | long        |
+---------+----------------+----------+-----------------------------------+-----------+-------------+
| 4576    | 66             | 63       | South Van Ness at Market          | 37.774814 | -122.418954 |
| 4607    | 10             | 70       | San Jose City Hall                | 37.337391 | -121.886995 |
| 4130    | 27             | 71       | Mountain View City Hall           | 37.389218 | -122.081896 |
| 4251    | 2              | 77       | San Jose Diridon Caltrain Station | 37.329732 | -121.901782 |
| 4299    | 66             | 83       | South Van Ness at Market          | 37.774814 | -122.418954 |
+---------+----------------+----------+-----------------------------------+-----------+-------------+


Для каждой стартовой станции найдем среднее время поездки. 

Рассчитаем среднее время поездки для каждого стартового парковочного места

In [21]:
spark.sql("""
SELECT
    start_station_name,
    AVG(duration) AS avg_duration
FROM trips
GROUP BY start_station_name
ORDER BY avg_duration DESC
""").show()


# Пример подготовки данных c Spark SQL, pandas, h3 для их визуализации на карте folium

In [22]:
# ! pip install h3 h3_pyspark pandas folium

Найдём велосипеды, которые ездили в рождество 2014 года.
https://spark.apache.org/docs/latest/api/sql/#make_timestamp

In [23]:
spark.sql("""
SELECT bike_id, start_date, end_date
FROM trips
WHERE start_date >= TIMESTAMP '2014-12-25 00:00:00'
  AND start_date < TIMESTAMP '2014-12-26 00:00:00'
ORDER BY start_date
""").show()


Найдём станции через которые проехал один из велосипедов, найденных ранее.

In [24]:
spark.sql("""
SELECT trips.bike_id, trips.start_date, trips.end_date, stations.name
FROM trips
INNER JOIN stations
    ON trips.start_station_id = stations.id
WHERE trips.bike_id = 583
  AND trips.start_date >= TIMESTAMP '2014-12-25 00:00:00'
  AND trips.start_date < TIMESTAMP '2014-12-26 00:00:00'
ORDER BY trips.start_date
""").show()


Найдём все станции, которые попали в ту же клетку h3 координатной сетки что и станции, через которые проехал велосипед 583 25.12.2014.

Отобразим gps координаты станций в координаты h3.

In [25]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import h3_pyspark
import h3

H3 Grid Resolutions https://h3geo.org/docs/core-library/restable/

In [26]:
resolution = 8

stationData.withColumn(
    'h3',
    h3_pyspark.geo_to_h3('lat', 'long', F.lit(resolution))
).createOrReplaceTempView("stations_h3")


Используя вложенный sql запрос, найдём h3 координаты станций, через который проехал велосипед 583. А затем отфильтруем поездки в рождество 2014 года, которые стартовали со станций с теми же h3 координатами, что мы нашли.

In [27]:
christmas_583_contacts = spark.sql("""
SELECT
    trips.bike_id,
    trips.start_date,
    stations_h3.h3,
    stations_h3.lat,
    stations_h3.long,
    stations_h3.name
FROM trips
INNER JOIN stations_h3
    ON trips.start_station_id = stations_h3.id
WHERE stations_h3.h3 IN (
        SELECT stations_h3.h3
        FROM trips
        INNER JOIN stations_h3
            ON trips.start_station_id = stations_h3.id
        WHERE trips.bike_id = 583
          AND trips.start_date >= TIMESTAMP '2014-12-25 00:00:00'
          AND trips.start_date < TIMESTAMP '2014-12-26 00:00:00'
    )
  AND trips.start_date >= TIMESTAMP '2014-12-25 00:00:00'
  AND trips.start_date < TIMESTAMP '2014-12-26 00:00:00'
ORDER BY trips.start_date
""")
christmas_583_contacts.cache()
christmas_583_contacts.show()


In [28]:
import pandas as pd
import h3

h3_places = christmas_583_contacts.select('lat','long', 'name', 'h3').toPandas()

In [29]:
# source code from https://nbviewer.org/github/uber/h3-py-notebooks/blob/master/notebooks/usage.ipynb
import folium 

def init_map(hexagons, width=1100, height=900):
    lats = []
    longs = []
    for hexagon in hexagons:
        lat, long = h3.h3_to_geo(hexagon)
        lats.append(lat)
        longs.append(long)
    return folium.Map(location=[sum(lats)/len(lats), sum(longs)/len(longs)], zoom_start=15, tiles='cartodbpositron', width=width, height=height)

def visualize_hexagons(folium_map, hexagons, color="red"):
    """
    hexagons is a list of hexcluster. Each hexcluster is a list of hexagons. 
    eg. [[hex1, hex2], [hex3, hex4]]
    """
    polylines = []
    lat = []
    lng = []
    for hex in hexagons:
        polygons = h3.h3_set_to_multi_polygon([hex], geo_json=False)
        # flatten polygons into loops.
        outlines = [loop for polygon in polygons for loop in polygon]
        polyline = [outline + [outline[0]] for outline in outlines][0]
        lat.extend(map(lambda v:v[0], polyline))
        lng.extend(map(lambda v:v[1], polyline))
        polylines.append(polyline)
    
    for polyline in polylines:
        my_PolyLine = folium.PolyLine(locations=polyline, weight=8, color=color)
        folium_map.add_child(my_PolyLine)
        
    return folium_map

def visualize_stations(folium_map, stations, color="red"):
    """
    stations is a dataframe with columns: lat, long, station_name
    """
    for idx, lat, long, station_name in stations.itertuples():
        folium_map.add_child(folium.map.Marker(location=(lat, long)))
        folium_map.add_child(folium.map.Marker(location=(lat, long), 
                        icon=folium.features.DivIcon(
                          icon_size=(500,36),
                          icon_anchor=(-17,37),
                          html=f'<div style="display: inline-block;font-size: 10pt; background: rgba(255, 255, 255, 0.8)">{station_name}</div>',
        )))
    return folium_map

In [30]:
m = init_map(h3_places.h3.unique())
visualize_hexagons(m, h3_places.h3.unique(), color="black")
visualize_stations(m, h3_places.loc[:, ['lat', 'long', 'name']])
display(m)

## Решение задач `L1_Apache_Spark_Tasks.md` через DataFrame API

In [31]:
from math import atan2, cos, radians, sin, sqrt
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

THREE_HOURS_SECONDS = 3 * 60 * 60

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0088
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return float(radius * c)

haversine_udf = F.udf(haversine_km, DoubleType())


In [32]:
bikeRuntime = tripData.groupBy("bike_id")\
    .agg(F.sum("duration").alias("total_duration"))\
    .orderBy(F.desc("total_duration"))

bikeRuntime.show(10)

maxBikeRow = bikeRuntime.first()
maxBikeId = int(maxBikeRow["bike_id"])
maxBikeTotalDuration = int(maxBikeRow["total_duration"])

stationsA = stationData.alias("a")
stationsB = stationData.alias("b")

maxDistance = stationsA.crossJoin(stationsB)\
    .where(F.col("a.id") < F.col("b.id"))\
    .withColumn(
        "distance_km",
        haversine_udf(F.col("a.lat"), F.col("a.long"), F.col("b.lat"), F.col("b.long"))
    )\
    .select(
        F.col("distance_km"),
        F.col("a.id").alias("station_1_id"),
        F.col("a.name").alias("station_1_name"),
        F.col("b.id").alias("station_2_id"),
        F.col("b.name").alias("station_2_name")
    )\
    .orderBy(F.desc("distance_km"))

maxDistance.show(1, truncate=False)


+-------------------+--------------+----------------------------+--------------+------------------------+
| distance_km       | station_1_id | station_1_name             | station_2_id | station_2_name         |
+-------------------+--------------+----------------------------+--------------+------------------------+
| 69.92125827585457 | 16           | SJSU - San Salvador at 9th | 60           | Embarcadero at Sansome |
+-------------------+--------------+----------------------------+--------------+------------------------+


In [33]:
maxBikeTrips = tripData.filter(F.col("bike_id") == maxBikeId)\
    .orderBy(F.col("start_date"), F.col("id"))

maxBikeTrips.select("id", "start_date", "start_station_name", "end_station_name", "duration").show(20, truncate=False)

maxBikePathRows = maxBikeTrips.select("start_station_name", "end_station_name").collect()
bikePath = "path not found" if not maxBikePathRows else " -> ".join(
    [maxBikePathRows[0]["start_station_name"]] + [row["end_station_name"] for row in maxBikePathRows]
)

bikesCount = tripData.select("bike_id").distinct().count()

heavyUsers = tripData\
    .filter(
        F.col("zip_code").isNotNull() &
        (F.trim(F.col("zip_code")) != "") &
        (F.lower(F.trim(F.col("zip_code"))) != "nil")
    )\
    .groupBy("zip_code")\
    .agg(F.sum("duration").alias("total_duration"))\
    .filter(F.col("total_duration") > THREE_HOURS_SECONDS)\
    .orderBy(F.desc("total_duration"))

heavyUsers.show(20, truncate=False)


+----------+----------------+
| zip_code | total_duration |
+----------+----------------+
| 94107    | 49757162       |
| 94105    | 25596128       |
| 94133    | 21637675       |
| 94102    | 19128021       |
| 94103    | 19127388       |
| 95531    | 17270400       |
| 94111    | 14244997       |
| 95112    | 12742370       |
| 94109    | 12057128       |
| 94040    | 7807926        |
| 94110    | 7421936        |
| 94117    | 6901313        |
| 94301    | 6590378        |
| 94041    | 6276284        |
| 94158    | 6248167        |
| 94306    | 5550643        |
| 94025    | 5178237        |
| 94108    | 5127562        |
| 94611    | 5014906        |
| 94010    | 4800158        |
+----------+----------------+


In [34]:
def format_duration(total_seconds):
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

maxDistanceRow = maxDistance.first()

print("1. Велосипед с максимальным временем пробега")
print(f"bikeId = {maxBikeId}")
print(f"Суммарное время = {format_duration(maxBikeTotalDuration)} ({maxBikeTotalDuration} сек.)")

print("\n2. Наибольшее геодезическое расстояние между станциями")
print(f"Станция 1: {maxDistanceRow['station_1_name']} (#{maxDistanceRow['station_1_id']})")
print(f"Станция 2: {maxDistanceRow['station_2_name']} (#{maxDistanceRow['station_2_id']})")
print(f"Расстояние = {maxDistanceRow['distance_km']:.3f} км")

print("\n3. Путь велосипеда с максимальным временем пробега через станции")
print(f"bikeId = {maxBikeId}")
print(f"Количество поездок этого велосипеда = {maxBikeTrips.count()}")
print(bikePath[:2000] + (" ..." if len(bikePath) > 2000 else ""))

print("\n4. Количество велосипедов в системе")
print(f"Количество велосипедов = {bikesCount}")

print("\n5. Пользователи, потратившие на поездки более 3 часов")
for row in heavyUsers.take(20):
    total_duration = int(row["total_duration"])
    print(f"zipCode = {row['zip_code']}, суммарное время = {format_duration(total_duration)} ({total_duration} сек.)")


1. Велосипед с максимальным временем пробега
bikeId = 535
Суммарное время = 5169:54:53 (18611693 сек.)

2. Наибольшее геодезическое расстояние между станциями
Станция 1: SJSU - San Salvador at 9th (#16)
Станция 2: Embarcadero at Sansome (#60)
Расстояние = 69.921 км

3. Путь велосипеда с максимальным временем пробега через станции
bikeId = 535
Количество поездок этого велосипеда = 1328
Post at Kearney -> San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome -> 2nd at South Park -> Davis at Jackson -> Civic Center BART (7th at Market) -> Post at Kearney -> Embarcadero at Sansome -> Washington at Kearney -> Market at Sansome -> Market at Sansome -> 2nd at Folsom -> 2nd at Townsend -> 2nd at Townsend -> Embarcadero at Sansome -> Clay at Battery -> Harry Bridges Plaza (Ferry Building) -> Clay at Battery -> San Francisco Caltrain (Townsend at 4th) -> Steuart at Market -> 2nd at Townsend -> Harry Bridges Plaza (Ferry Building) -> Townsend at 

In [35]:
sc.stop()